In [1]:
import pandas as pd
import numpy as np
from random_strategy_benchmark.backtester import calculate_return
from random_strategy_benchmark.strategies import run_momentum_strategy, generate_momentum_signal

In [2]:
df = pd.read_csv("../../data/spy_daily.csv")
prices = df["Adj Close"]
returns = calculate_return(prices)
lookback = 252
momentum_df, momentum_results = run_momentum_strategy(prices, returns, lookback, 0.001)
signal = generate_momentum_signal(prices, lookback)
transaction_cost = 0.001

returns_array = np.array(returns)

In [3]:
signal_array = np.array(signal)

position_array = np.empty_like(signal_array)

position_array[0] = np.nan
position_array[1:] = signal_array[:-1]

strategy_return_array = returns_array * position_array

df = pd.DataFrame({
    "position": position_array,
    "strategy_return": strategy_return_array
})

In [4]:
days_invested = (position_array == 1).sum()
valid_days = (~np.isnan(position_array)).sum()
time_invested = days_invested / valid_days

position_filled_array = np.where(np.isnan(position_array), 0, position_array)

previous_position_array = np.empty_like(position_array)

previous_position_array[1:] = position_filled_array[:-1]
previous_position_array[0] = 0

trade_indicator = (position_filled_array != previous_position_array)
trade_count = trade_indicator.sum()

In [5]:
# Transaction Costs

transaction_cost_array = trade_indicator * transaction_cost
net_strategy_return_array = strategy_return_array - transaction_cost_array

# Equity

valid_list = ~np.isnan(net_strategy_return_array)

if not valid_list.any():
    raise ValueError("No valid strategy returns were generated.")

first_valid = np.argmax(valid_list)

initial_equity = 1
start_index = first_valid - 1

equity_array = np.empty_like(net_strategy_return_array)

equity_array[:start_index] = np.nan
equity_array[start_index] = initial_equity

valid_returns = net_strategy_return_array[first_valid:]

compounded_equity = np.cumprod(1 + valid_returns)

equity_array[first_valid:] = compounded_equity

final_equity = equity_array[-1]

total_return = (final_equity / initial_equity) - 1

performance_return = pd.Series(net_strategy_return_array, index = df.index)
df["equity"] = pd.Series(equity_array, index = df.index)

In [6]:
risk_free_rate = 0

daily_volatility = np.nanstd(net_strategy_return_array, ddof=1)
mean_daily_return = np.nanmean(net_strategy_return_array)

sharpe_daily = (mean_daily_return - risk_free_rate)/daily_volatility

annual_volatility = np.sqrt(252) * daily_volatility
sharpe_ratio = np.sqrt(252) * sharpe_daily

In [7]:
running_peak_array = np.maximum.accumulate(equity_array)

drawdown_array = equity_array / running_peak_array - 1
max_drawdown = np.min(drawdown_array)

In [8]:
growth_multiple = final_equity / initial_equity
years = valid_days / 252
cagr = growth_multiple ** (1 / years) - 1
calmar_ratio = cagr / abs(max_drawdown)